In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

data_root = Path("data")
sample_id = "46"

a = np.asarray(
    Image.open(data_root / "A" / f"{sample_id}.png").convert("RGB"),
    dtype=np.uint8
)

b = np.asarray(
    Image.open(data_root / "B" / f"{sample_id}.png").convert("RGB"),
    dtype=np.uint8
)

assert a.shape == (96, 96, 3)
assert b.shape == (96, 96, 3)


def write_array(f, name, arr):
    flat = arr.reshape(-1)

    f.write(f"const uint8_t {name}[{len(flat)}] = {{\n")

    for i in range(0, len(flat), 16):
        vals = ", ".join(str(v) for v in flat[i:i + 16])
        f.write(f"    {vals},\n")

    f.write("};\n\n")


with open("../firmware/BTC_STM/STM32CubeIDE/Appli/Application/User/Core/images.c", "w") as f:
    f.write('#include "images.h"\n\n')

    write_array(f, "image_a_rgb", a)
    write_array(f, "image_b_rgb", b)

with open("../firmware/BTC_STM/Appli/Core/Inc/images.h", "w") as f:
    f.write("""#ifndef IMAGES_H
#define IMAGES_H

#include <stdint.h>

#define IMAGE_W 96
#define IMAGE_H 96
#define IMAGE_C 3

extern const uint8_t image_a_rgb[IMAGE_W * IMAGE_H * IMAGE_C];
extern const uint8_t image_b_rgb[IMAGE_W * IMAGE_H * IMAGE_C];

#endif
""")